In [10]:
# ── Verify your real key is now set correctly ─────────────────
from google.colab import userdata
import os

# Re-fetch from Secrets after your update
api_key = userdata.get("OPENAI_API_KEY")

print(f"Key preview: {api_key[:8]}...{api_key[-4:] if api_key else 'EMPTY'}")
print(f"Key length : {len(api_key) if api_key else 0} chars")

# A real key looks like: sk-proj-AbCdEf... (length 50-170 chars)
# A fake key looks like: sk-your-actual-key-here (length 22 chars)

if not api_key:
    print("❌ Still empty — make sure you saved in Colab Secrets")
elif "your" in api_key.lower() or "placeholder" in api_key.lower():
    print("❌ Still a placeholder — you need to paste your REAL key from platform.openai.com")
elif len(api_key) < 30:
    print("❌ Too short to be real — real keys are 50+ characters")
elif not api_key.startswith("sk-"):
    print("❌ Must start with 'sk-' — check you copied the full key")
else:
    print("✅ Key format looks correct — testing connection...")
    os.environ["OPENAI_API_KEY"] = api_key

    from openai import OpenAI
    try:
        client = OpenAI(api_key=api_key)
        test = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Say READY"}],
            max_tokens=5
        )
        print(f"✅ Connected! Response: {test.choices[0].message.content}")
        print(f"✅ You can now run all remaining cells")
    except Exception as e:
        print(f"❌ Error: {e}")

Key preview: sk-proj-...BLUA
Key length : 164 chars
✅ Key format looks correct — testing connection...
❌ Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [18]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "groq"])
print("✅ groq library installed.")

✅ groq library installed.


In [14]:
# ============================================================
# CELL 1 — Using GROQ (Free, Fast, No credit card)
# ============================================================
import os, json
from google.colab import userdata
from groq import Groq
from tenacity import retry, stop_after_attempt, wait_exponential
from dataclasses import dataclass, field
from typing import List

# Install groq
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "groq"])

# Load key
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Groq key loaded from Colab Secrets")
except:
    os.environ["GROQ_API_KEY"] = input("Paste your Groq API key: ").strip()

# Connect and test
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL  = "llama-3.3-70b-versatile"   # Free, extremely capable

try:
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Say READY"}],
        max_tokens=5
    )
    print(f"✅ Groq connected  →  model={MODEL}")
    print(f"✅ Test response: {test.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# ── All agent functions (same API, just different client) ─────

@dataclass
class RewrittenQuery:
    original_query: str
    sub_queries: List[str]     = field(default_factory=list)
    expanded_query: str        = ""
    hypothetical_answer: str   = ""
    strategy_used: str         = "standard"

    @property
    def all_queries(self):
        queries = [self.original_query, self.expanded_query]
        queries.extend(self.sub_queries)
        if self.hypothetical_answer:
            queries.append(self.hypothetical_answer)
        return list(dict.fromkeys(q for q in queries if q))


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=8))
def rewrite_decompose(query: str) -> RewrittenQuery:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""
You are a search query optimizer. Decompose this query into focused sub-queries.
Query: "{query}"

Return only valid JSON, no markdown, no explanation:
{{
  "sub_queries": ["specific sub-query 1", "specific sub-query 2", "specific sub-query 3"],
  "expanded_query": "expanded version with synonyms and related terms",
  "step_back": "broader context question"
}}"""}],
        temperature=0.3,
        response_format={"type": "json_object"}
    )
    data = json.loads(response.choices[0].message.content)
    return RewrittenQuery(
        original_query=query,
        sub_queries=data.get("sub_queries", []),
        expanded_query=data.get("expanded_query", query),
        strategy_used="sub_query_decomposition"
    )


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=8))
def rewrite_hyde(query: str) -> RewrittenQuery:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""
Write a 100-150 word factual paragraph that perfectly answers this question.
Write as if from an authoritative textbook. No preamble, just the paragraph.

Question: {query}"""}],
        temperature=0.4
    )
    return RewrittenQuery(
        original_query=query,
        hypothetical_answer=response.choices[0].message.content.strip(),
        expanded_query=query,
        strategy_used="hyde"
    )


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=8))
def rewrite_multi_query(query: str) -> RewrittenQuery:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""
Generate 4 different phrasings of this search query, each from a different angle.
Return only valid JSON: {{"queries": ["phrasing 1", "phrasing 2", "phrasing 3", "phrasing 4"]}}

Original: "{query}" """}],
        temperature=0.7,
        response_format={"type": "json_object"}
    )
    data = json.loads(response.choices[0].message.content)
    queries = data.get("queries", [query])
    return RewrittenQuery(
        original_query=query,
        sub_queries=queries,
        expanded_query=queries[0] if queries else query,
        strategy_used="multi_query"
    )


def smart_rewrite(query: str) -> RewrittenQuery:
    q = query.lower()
    if any(w in q for w in ["how", "why", "explain", "describe", "what causes"]):
        return rewrite_decompose(query)
    elif len(query.split()) < 4:
        return rewrite_multi_query(query)
    elif "?" in query:
        return rewrite_hyde(query)
    return rewrite_decompose(query)


print("\n✅ All agent functions ready")
print("   → rewrite_decompose(query)")
print("   → rewrite_hyde(query)")
print("   → rewrite_multi_query(query)")
print("   → smart_rewrite(query)")
print("\n▶  Run Cell 2 to test all strategies")

✅ Groq key loaded from Colab Secrets
✅ Groq connected  →  model=llama-3.3-70b-versatile
✅ Test response: READY

✅ All agent functions ready
   → rewrite_decompose(query)
   → rewrite_hyde(query)
   → rewrite_multi_query(query)
   → smart_rewrite(query)

▶  Run Cell 2 to test all strategies


In [15]:
test_query = "How does attention mechanism work in transformer models?"
print(f"Original: {test_query}\n")

print("=" * 50)
print("Strategy 1: Sub-Query Decomposition")
result1 = rewrite_decompose(test_query)
print(f"Sub-queries: {result1.sub_queries}")
print(f"Expanded: {result1.expanded_query}")

print("\n" + "=" * 50)
print("Strategy 2: HyDE")
result2 = rewrite_hyde(test_query)
print(f"Hypothetical answer (first 300 chars):")
print(result2.hypothetical_answer[:300])

print("\n" + "=" * 50)
print("Strategy 3: Multi-Query")
result3 = rewrite_multi_query(test_query)
print(f"Generated queries: {result3.sub_queries}")

print("\n" + "=" * 50)
print("Auto Strategy Selection")
auto_result = smart_rewrite(test_query)
print(f"Selected strategy: {auto_result.strategy_used}")
print(f"All queries: {auto_result.all_queries}")

Original: How does attention mechanism work in transformer models?

Strategy 1: Sub-Query Decomposition
Sub-queries: ['What is the role of attention in transformer architecture?', 'How does self-attention mechanism work in transformers?', 'What are the benefits of using attention mechanism in transformer models?']
Expanded: How do attention mechanisms, including self-attention and multi-head attention, function within transformer models, and what are their applications and advantages in natural language processing tasks

Strategy 2: HyDE
Hypothetical answer (first 300 chars):
The attention mechanism in transformer models is a key component that enables the model to focus on specific parts of the input sequence when generating output. It works by computing attention weights based on the query, key, and value vectors, which are derived from the input sequence. The query ve

Strategy 3: Multi-Query
Generated queries: ['What is the role of attention in transformer architecture?', 'Transfor

In [16]:
from dataclasses import dataclass
from typing import List

@dataclass
class RetrievalEvaluation:
    overall_score: float
    individual_scores: List[float]
    is_sufficient: bool
    reasoning: str
    suggested_action: str
    filtered_results: list = None

    def __post_init__(self):
        if self.filtered_results is None:
            self.filtered_results = []

QUALITY_THRESHOLD = float(os.environ.get("RETRIEVAL_QUALITY_THRESHOLD", "0.5"))

def evaluate_retrieval(query: str, results: list) -> RetrievalEvaluation:
    """LLM-based retrieval quality evaluator."""
    if not results:
        return RetrievalEvaluation(
            overall_score=0.0, individual_scores=[],
            is_sufficient=False, reasoning="No results",
            suggested_action="fallback"
        )

    # Format top 5 results
    context = ""
    for i, r in enumerate(results[:5]):
        context += f"\n[Doc {i+1}]: {r.content[:300]}...\n"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""
You are a retrieval quality evaluator.

Query: "{query}"

Retrieved Documents:
{context}

Rate each document's relevance (0.0-1.0) and overall quality.
Return JSON (no markdown):
{{
  "individual_scores": [0.8, 0.5, 0.9, 0.3, 0.7],
  "overall_score": 0.64,
  "is_sufficient": true,
  "reasoning": "Brief reason",
  "suggested_action": "proceed"
}}

suggested_action: "proceed" (good), "requery" (try again), "fallback" (no good docs found)"""}],
        temperature=0.1,
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    overall = data.get("overall_score", 0.5)
    ind_scores = data.get("individual_scores", [0.5] * min(5, len(results)))

    # Filter to keep relevant results only
    filtered = []
    for r, score in zip(results[:5], ind_scores):
        if score >= 0.4:
            r.score = score
            filtered.append(r)
    filtered.extend(results[5:])

    return RetrievalEvaluation(
        overall_score=overall,
        individual_scores=ind_scores,
        is_sufficient=data.get("is_sufficient", overall >= QUALITY_THRESHOLD),
        reasoning=data.get("reasoning", ""),
        suggested_action=data.get("suggested_action", "proceed"),
        filtered_results=filtered
    )

print("✅ Retrieval Evaluator ready")

✅ Retrieval Evaluator ready


In [20]:
import uuid

@dataclass
class SearchResult:
    chunk_id: str
    content: str
    score: float = 0.0

def hybrid_search(query: str, top_k: int = 5) -> List[SearchResult]:
    """Placeholder for a hybrid search function.
    In a real RAG system, this would query a document store (e.g., vector DB)
    and return relevant document chunks.
    """
    print(f"Performing hybrid search for: '{query}' (top_k={top_k})")

    # This is a placeholder implementation. Replace with actual search logic.
    dummy_results = [
        SearchResult(
            chunk_id=str(uuid.uuid4()),
            content=f"This is a dummy document about RAG systems challenges for query '{query}'. It discusses issues like data freshness, latency, and hallucination."
        ),
        SearchResult(
            chunk_id=str(uuid.uuid4()),
            content=f"Another dummy document related to RAG systems and their architectural complexities for query '{query}'. It highlights the need for robust indexing and retrieval components."
        ),
        SearchResult(
            chunk_id=str(uuid.uuid4()),
            content=f"A third dummy document focusing on the evaluation metrics for RAG systems for query '{query}'. It mentions precision, recall, and contextual relevance."
        )
    ]

    # Filter based on top_k, if we had more dummy results
    return dummy_results[:top_k]

print("✅ `hybrid_search` function defined (placeholder implementation)")

✅ `hybrid_search` function defined (placeholder implementation)


In [21]:
query = "What are the main challenges in building RAG systems?"

print(f"Query: {query}\n")

# Step 1: Rewrite
rewritten = smart_rewrite(query)
print(f"Strategy: {rewritten.strategy_used}")
print(f"Sub-queries: {rewritten.sub_queries}\n")

# Step 2: Retrieve (using all sub-queries)
all_results = []
seen_ids = set()
for sub_q in rewritten.all_queries[:3]:
    for r in hybrid_search(sub_q, top_k=10):
        if r.chunk_id not in seen_ids:
            seen_ids.add(r.chunk_id)
            all_results.append(r)

print(f"Retrieved: {len(all_results)} unique chunks\n")

# Step 3: Evaluate
evaluation = evaluate_retrieval(query, all_results)
print(f"Evaluation Results:")
print(f"  Overall Score: {evaluation.overall_score:.3f}")
print(f"  Is Sufficient: {evaluation.is_sufficient}")
print(f"  Suggested Action: {evaluation.suggested_action}")
print(f"  Reasoning: {evaluation.reasoning}")
print(f"  Kept after filtering: {len(evaluation.filtered_results)} chunks")

Query: What are the main challenges in building RAG systems?

Strategy: hyde
Sub-queries: []

Performing hybrid search for: 'What are the main challenges in building RAG systems?' (top_k=10)
Performing hybrid search for: 'The development of Retrieval-Augmented Generation (RAG) systems poses several key challenges. One primary concern is the need for large, high-quality datasets to train both the retriever and generator components. Additionally, RAG systems require careful tuning of hyperparameters to balance the trade-off between retrieval and generation, as well as effective integration of the two components. Another significant challenge is addressing the potential for hallucination, where the generator produces inaccurate or nonsensical content. Furthermore, RAG systems must also contend with issues related to efficiency, scalability, and interpretability, particularly as the size and complexity of the input data increase. Finally, ensuring the robustness and reliability of RAG syst